# LoRA fine-tune: Qwen 3.5 9B for the knowledge workflow pipeline

Trains a LoRA adapter on examples built from an ontology suite (BFO, CCO, RO, QUDT) in the pipeline's own prompt and JSON formats, then exports a 4-bit GGUF for Ollama.

**Before you start**
1. On your machine: `python -m src.run lora-data` → creates `outputs/lora/kw_lora_upload.zip`.
2. Colab: **Runtime → Change runtime type**. An **L4 or A100** (24 GB+, Colab Pro) trains the 9B, which matches your local model. An L4 with less memory trains the **4B**. A free **T4** has no bf16, so it can only train the **2B** (float32). Qwen 3.5 9B needs about 22 GB at bf16, and Unsloth advises against 4-bit QLoRA for Qwen 3.5.
3. Run the cells in order. The last cell copies the adapter and GGUF to Google Drive.

In [ ]:
# 1. GPU check: 24 GB+ (L4, A100) trains the 9B; a T4 (no bf16) trains the 2B in float32
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.get_device_capability(0)[0] >= 8  # native bf16 (Ampere+); a T4 only emulates it, which the trainer rejects
print(f"{GPU_GB:.0f} GB GPU memory, bf16 {'yes' if BF16 else 'NO: only the 2B can be trained (float32); use an L4/A100 for 4B/9B'}")
assert GPU_GB >= 14, "Need at least a T4 (16 GB). Change the runtime type."

In [ ]:
# 2. Install Unsloth WITHOUT replacing Colab's own PyTorch/torchvision (replacing them breaks torchvision
#    and can pull a CUDA build the Colab driver doesn't support). Run this on a FRESH runtime:
#    if an earlier install already changed torch, use Runtime -> Disconnect and delete runtime first.
import torch, torchvision
PINS = f"torch=={torch.__version__.split('+')[0]} torchvision=={torchvision.__version__.split('+')[0]}"
print("keeping Colab's", PINS, "| CUDA", torch.version.cuda)
%pip install -q unsloth unsloth_zoo {PINS}

In [ ]:
# 3. Settings. SIZE is picked from the GPU; set it by hand to override ("9B" ~22 GB bf16, "4B" ~10 GB bf16,
#    "2B" in float32 on GPUs without bf16).
# Qwen 3.5 cannot train in float16. Without bf16 (T4, P100) the only option is float32, which fits the 2B only.
SIZE = ("9B" if GPU_GB >= 22 else "4B") if BF16 else "2B"
MODEL_NAME = f"Qwen/Qwen3.5-{SIZE}"
MAX_SEQ = {"9B": 8192, "4B": 6144, "2B": 4096}[SIZE]   # longer examples are dropped (cell 5 prints how many)
LORA_R, LORA_ALPHA = 16, 16
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH, GRAD_ACCUM = 1, 8
RUN_NAME = f"kw-qwen3.5-{SIZE.lower()}-lora"
DRIVE_DIR = "/content/drive/MyDrive/kw_lora"
print(MODEL_NAME, "->", RUN_NAME)

In [ ]:
# 4. Upload kw_lora_upload.zip (or put it in Drive and set ZIP_PATH)
import os, zipfile
ZIP_PATH = None  # e.g. "/content/drive/MyDrive/kw_lora/kw_lora_upload.zip"
if ZIP_PATH is None:
    from google.colab import files
    ZIP_PATH = next(iter(files.upload()))
zipfile.ZipFile(ZIP_PATH).extractall("/content/kw_lora")
print(os.listdir("/content/kw_lora/data"))
print(open("/content/kw_lora/data/dataset_card.json").read()[:1500])

In [ ]:
# 5. Load the model (bf16 LoRA) and format the data with the chat template, thinking off
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"  # text-only training never uses torchvision
if not BF16:  # no bf16 (T4): keep Unsloth from autocasting to float16, which Qwen 3.5 cannot use
    os.environ["UNSLOTH_FORCE_FLOAT32"] = "1"
from unsloth import FastLanguageModel
from datasets import load_dataset

import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=False, full_finetuning=False,
    **({"load_in_16bit": True} if BF16 else {"dtype": torch.float32}),  # never float16 for Qwen 3.5
)
tok = getattr(tokenizer, "tokenizer", tokenizer)

def to_text(example):
    try:
        text = tokenizer.apply_chat_template(example["messages"], tokenize=False, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text, "n_tokens": len(tok(text).input_ids)}

data = load_dataset("json", data_files={s: f"/content/kw_lora/data/{s}.jsonl" for s in ("train", "val", "test")})
data = data.map(to_text)
for split in ("train", "val"):
    before = len(data[split])
    data[split] = data[split].filter(lambda e: e["n_tokens"] <= MAX_SEQ)
    print(f"{split}: kept {len(data[split])}/{before} examples (<= {MAX_SEQ} tokens)")
from collections import Counter
print("train tasks:", Counter(data["train"]["task"]))
print(data["train"][0]["text"][-600:])

In [ ]:
# 6. LoRA adapter
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407, max_seq_length=MAX_SEQ,
)
if not BF16:  # some Qwen 3.5 weights load in bf16 despite dtype=float32; mixing them fails, so cast all to float32
    import torch
    for t in list(model.parameters()) + list(model.buffers()):
        if t.dtype in (torch.bfloat16, torch.float16):
            t.data = t.data.float()
    print("dtypes:", {str(t.dtype) for t in model.parameters()})

In [ ]:
# 7. Train on the assistant replies only
import math, torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

WARMUP = max(1, math.ceil(0.03 * EPOCHS * len(data["train"]) / (BATCH * GRAD_ACCUM)))

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=data["train"], eval_dataset=data["val"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_steps=WARMUP,
        logging_steps=5, eval_strategy="steps", eval_steps=50, save_strategy="steps", save_steps=100,
        optim="adamw_8bit", weight_decay=0.01, bf16=BF16, fp16=False,  # Qwen 3.5 overflows in float16; float32 when no bf16
        seed=3407,
        output_dir=f"/content/{RUN_NAME}-checkpoints", report_to="none", dataset_num_proc=1,
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

In [ ]:
# 8. OPTIONAL quick sanity check (slow on a T4; skip it and go to cell 9 if you like).
#    The real evaluation is `lora-eval` on your machine.
SANITY_EXAMPLES = 2
import json, re
FastLanguageModel.for_inference(model)

def reply(messages):
    try:
        prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=768, do_sample=False)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

for ex in data["test"].shuffle(seed=1).select(range(min(SANITY_EXAMPLES, len(data["test"])))):
    text = reply(ex["messages"])
    body = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    try:
        json.loads(body[body.find("{"): body.rfind("}") + 1]); ok = "valid JSON"
    except Exception:
        ok = "INVALID JSON"
    print(f"[{ex['task']}] {ok}: {body[:300]}\n")

In [ ]:
# 9. Save the adapter and a 4-bit GGUF, and copy both to Google Drive
import glob, shutil
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

# latest training checkpoint to Drive first (a safety copy of the adapter)
ckpts = sorted(glob.glob(f"/content/{RUN_NAME}-checkpoints/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
if ckpts:
    shutil.copytree(ckpts[-1], f"{DRIVE_DIR}/{os.path.basename(ckpts[-1])}", dirs_exist_ok=True)
# the adapter itself, saved straight from the LoRA weights
from peft import get_peft_model_state_dict
from safetensors.torch import save_file
lora = get_peft_model_state_dict(model)
lost = [k for k, v in lora.items() if v.device.type == "meta"]
assert not lost, f"{len(lost)} LoRA tensors are no longer in memory; use the recovery cell with the checkpoint"
ADAPTER_DIR = f"{DRIVE_DIR}/{RUN_NAME}-adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)
save_file({k: v.detach().to("cpu").contiguous() for k, v in lora.items()}, f"{ADAPTER_DIR}/adapter_model.safetensors")
model.peft_config["default"].save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to", ADAPTER_DIR)

model.save_pretrained_gguf(f"/content/{RUN_NAME}-gguf", tokenizer, quantization_method="q4_k_m")
gguf = [f for f in glob.glob(f"/content/{RUN_NAME}-gguf*/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf")
        if "q4_k_m" in f.lower()]
print(gguf)
target = f"{DRIVE_DIR}/{RUN_NAME}.Q4_K_M.gguf"
shutil.copy(gguf[0], target)
json.dump({"base": MODEL_NAME, "r": LORA_R, "alpha": LORA_ALPHA, "epochs": EPOCHS, "lr": LEARNING_RATE,
           "max_seq": MAX_SEQ, "train_examples": len(data["train"]), "val_examples": len(data["val"]),
           "train_loss": stats.training_loss, "log_history": trainer.state.log_history},
          open(f"{DRIVE_DIR}/{RUN_NAME}-training.json", "w"), indent=1)
print("saved to", DRIVE_DIR)

### Recovery: export from a saved adapter
Use this if cell 9 failed (for example `Cannot copy out of meta tensor` after interrupting cell 8), or to re-export later.
**Runtime → Restart session**, run cells 1 and 3 (the packages from cell 2 are still installed), then this cell.

In [ ]:
# 10. Rebuild the model from the saved adapter (or a training checkpoint) and export the 4-bit GGUF
import os, glob, json, shutil
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
if not BF16:
    os.environ["UNSLOTH_FORCE_FLOAT32"] = "1"
from google.colab import drive
drive.mount("/content/drive")
from unsloth import FastLanguageModel

ADAPTER = f"{DRIVE_DIR}/{RUN_NAME}-adapter"
if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):  # fall back to the newest checkpoint copied to Drive
    ADAPTER = sorted(glob.glob(f"{DRIVE_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))[-1]
print("exporting from", ADAPTER)
import torch
model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER, max_seq_length=MAX_SEQ, load_in_4bit=False,
                                                     **({"load_in_16bit": True} if BF16 else {"dtype": torch.float32}))
model.save_pretrained_gguf(f"/content/{RUN_NAME}-gguf", tokenizer, quantization_method="q4_k_m")
gguf = [f for f in glob.glob(f"/content/{RUN_NAME}-gguf*/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf")
        if "q4_k_m" in f.lower()]
shutil.copy(gguf[0], f"{DRIVE_DIR}/{RUN_NAME}.Q4_K_M.gguf")
print("saved", f"{DRIVE_DIR}/{RUN_NAME}.Q4_K_M.gguf")

## Back on your machine (2B from a T4)

1. Download `kw-qwen3.5-2b-lora.Q4_K_M.gguf` and `kw-qwen3.5-2b-lora-training.json` from Drive (`MyDrive/kw_lora`) into `opus_knowledge_workflow/outputs/lora/`.
2. From `opus_knowledge_workflow`, build the fine-tuned model and the base 2B to compare against (same 32K context):
   ```
   ollama create kw-qwen3.5-2b-lora-32k -f src/resources/ollama/Modelfile.lora
   ollama pull qwen3.5:2b
   ollama create kw-qwen3.5-2b-32k -f src/resources/ollama/Modelfile.2b
   ```
3. Compare base vs fine-tuned on the same held-out test set:
   ```
   uv run --with-requirements requirements.txt python -m src.run lora-eval --model kw-qwen3.5-2b-32k
   uv run --with-requirements requirements.txt python -m src.run lora-eval --model kw-qwen3.5-2b-lora-32k
   ```
   Results: `outputs/lora/eval_summary.csv`. Add `--model kw-qwen3.5-9b-32k` to see how the 2B LoRA compares with your current 9B.
4. Use it in the pipeline: `LLM_PROFILE = "ollama-lora"` in `src/config.py`.
